In [1]:
# Cell 1
import time
from photonicdrivers.Magnets.APS100_PS_Stub import APS100_PS_Stub

stub = APS100_PS_Stub(IP_address="127.0.0.1", IP_port=4444)
stub.connect()
stub.set_control_remote()
print("connected:", stub.is_connected())
print("id:", stub.get_id())
print("mode:", stub.get_control_mode())
print("unit:", stub.get_unit())
print("channel:", stub.get_channel())
print("field:", stub.get_field())

Stub connection will simulate Ethernet to 127.0.0.1:4444
Stub: Connected
connected: True
id: Stub APS100 Power Supply
mode: remote
unit: kG
channel: 1
field: 0.0


In [2]:


# Cell 2

stub.set_unit("kG")
stub.set_channel(1)
stub.set_lower_limit(0.0, "kG")
stub.set_upper_limit(2.0, "kG")

print("lower:", stub.get_lower_limit())
print("upper:", stub.get_upper_limit())
print("sweep mode:", stub.get_sweep_mode())

# Cell 3
start = time.perf_counter()
stub.ramp_up(wait_while_ramping=False)
elapsed = time.perf_counter() - start

print("returned in:", elapsed, "s")
print("mode immediately after start:", stub.get_sweep_mode())
print("field immediately after start:", stub.get_field())

# Cell 4
stub.ramp_down(wait_while_ramping=False)

for _ in range(20):
    print("mode:", stub.get_sweep_mode(), "field:", stub.get_field())
    if stub.get_sweep_mode() == "Standby":
        break
    time.sleep(0.2)

# Cell 5
stub.set_channel(2)
stub.set_upper_limit(1.5, "kG")
stub.ramp_up(wait_while_ramping=False)

for _ in range(20):
    print("ch2 mode:", stub.get_sweep_mode(), "field:", stub.get_field())
    if stub.get_sweep_mode() == "Standby":
        break
    time.sleep(0.2)

print("channel 2 final:", stub.get_field())
stub.set_channel(1)
print("channel 1 final:", stub.get_field())

lower: 0.0kG
upper: (2.0, 'kG')
sweep mode: Standby
returned in: 0.0005112000071676448 s
mode immediately after start: Sweeping up
field immediately after start: 0.0125
mode: Standby field: 0.0
ch2 mode: Sweeping up field: 0.0125
ch2 mode: Sweeping up field: 0.05
ch2 mode: Sweeping up field: 0.09999999999999999
ch2 mode: Sweeping up field: 0.15
ch2 mode: Sweeping up field: 0.20000000000000004
ch2 mode: Sweeping up field: 0.25000000000000006
ch2 mode: Sweeping up field: 0.3000000000000001
ch2 mode: Sweeping up field: 0.35000000000000014
ch2 mode: Sweeping up field: 0.4000000000000002
ch2 mode: Sweeping up field: 0.45000000000000023
ch2 mode: Sweeping up field: 0.5000000000000002
ch2 mode: Sweeping up field: 0.55
ch2 mode: Sweeping up field: 0.5999999999999999
ch2 mode: Sweeping up field: 0.6499999999999997
ch2 mode: Sweeping up field: 0.6999999999999995
ch2 mode: Sweeping up field: 0.7499999999999993
ch2 mode: Sweeping up field: 0.7999999999999992
ch2 mode: Sweeping up field: 0.84999999

In [4]:
# Cell 6: Verify intermediate fields during ramping
def capture_ramp_trace(stub, start_cmd, poll_s=0.05, timeout_s=10.0):
    """Capture (time, field, mode) samples from ramp start until Standby."""
    t0 = time.perf_counter()
    _ = start_cmd()
    samples = []
    while True:
        t = time.perf_counter() - t0
        mode = stub.get_sweep_mode()
        field = stub.get_field()
        samples.append((t, field, mode))
        if mode == "Standby":
            break
        if t > timeout_s:
            raise TimeoutError(f"Ramp did not finish within {timeout_s}s")
        time.sleep(poll_s)
    return samples

def assert_intermediate_quality(samples, lower, upper, increasing=True, eps=1e-9):
    fields = [s[1] for s in samples]
    assert len(fields) >= 3, f"Need at least 3 samples, got {len(fields)}"
    assert all((lower - eps) <= f <= (upper + eps) for f in fields), (
        f"Out-of-bounds field sample found. Allowed [{lower}, {upper}], got min={min(fields)}, max={max(fields)}"
    )
    if increasing:
        assert all(fields[i+1] >= fields[i] - eps for i in range(len(fields)-1)), (
            "Field is not monotonic non-decreasing during ramp up"
        )
    else:
        assert all(fields[i+1] <= fields[i] + eps for i in range(len(fields)-1)), (
            "Field is not monotonic non-increasing during ramp down"
        )
    # Ensure we observed true intermediate points, not only endpoints.
    assert any((lower + 1e-6) < f < (upper - 1e-6) for f in fields), (
        "No strict intermediate field sample observed between endpoints"
    )
    return fields

# Configure deterministic test window on channel 1
stub.set_channel(1)
stub.set_unit("kG")
stub.set_lower_limit(0.0, "kG")
stub.set_upper_limit(2.0, "kG")
stub.set_ramp_rate(1.0)  # Slower ramp gives richer intermediate sampling

# Start from a known state
stub.ramp_to_zero(wait_while_ramping=True)

# Ramp up and verify intermediate behavior
up_samples = capture_ramp_trace(stub, lambda: stub.ramp_up(wait_while_ramping=False))
up_fields = assert_intermediate_quality(up_samples, lower=0.0, upper=2.0, increasing=True)

# Ramp down and verify intermediate behavior
down_samples = capture_ramp_trace(stub, lambda: stub.ramp_down(wait_while_ramping=False))
down_fields = assert_intermediate_quality(down_samples, lower=0.0, upper=2.0, increasing=False)

print("Intermediate-field checks passed")
print(f"Ramp-up samples: {len(up_fields)}, first={up_fields[0]:.3f}, mid={up_fields[len(up_fields)//2]:.3f}, last={up_fields[-1]:.3f}")
print(f"Ramp-down samples: {len(down_fields)}, first={down_fields[0]:.3f}, mid={down_fields[len(down_fields)//2]:.3f}, last={down_fields[-1]:.3f}")

Intermediate-field checks passed
Ramp-up samples: 40, first=0.050, mid=1.050, last=2.000
Ramp-down samples: 41, first=1.950, mid=0.950, last=0.000
